# PDD Mobility Scanner — Slope Analysis

Step-by-step slope computation from `trail_data.csv` using gravity vector and GPS data.

**Upload your `trail_data.csv` when prompted.**

## Step 1: Load the CSV

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"Loaded {len(df)} rows, {df['wp'].nunique()} waypoints")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 2: Relative height from GPS altitude

Zero the GPS altitude to the starting point so each waypoint shows how far above or below the start you are.

In [ ]:
# Get one altitude per waypoint (last GPS reading in each segment)
wp_alt = df.dropna(subset=['alt']).groupby('wp').agg(
    alt=('alt', 'last'),
    lat=('lat', 'last'),
    lng=('lng', 'last'),
    samples=('ms', 'count')
).reset_index()

# Zero to starting altitude
baseline_alt = wp_alt['alt'].iloc[0]
wp_alt['height_m'] = wp_alt['alt'] - baseline_alt

print(f"Baseline altitude: {baseline_alt:.1f}m")
print(f"Height range: {wp_alt['height_m'].min():.1f}m to {wp_alt['height_m'].max():.1f}m")
print()
wp_alt[['wp', 'alt', 'height_m', 'lat', 'lng']]

## Step 3: Distance and grade between waypoints

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_m(lat1, lon1, lat2, lon2):
    """Distance in meters between two GPS points."""
    R = 6371000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Distance and height change between consecutive waypoints
wp_alt['dist_m'] = 0.0
wp_alt['height_change_m'] = 0.0
wp_alt['grade_pct'] = 0.0

for i in range(1, len(wp_alt)):
    idx = wp_alt.index[i]
    prev = wp_alt.index[i-1]
    d = haversine_m(wp_alt.loc[prev, 'lat'], wp_alt.loc[prev, 'lng'],
                    wp_alt.loc[idx, 'lat'], wp_alt.loc[idx, 'lng'])
    h = wp_alt.loc[idx, 'height_m'] - wp_alt.loc[prev, 'height_m']
    wp_alt.loc[idx, 'dist_m'] = d
    wp_alt.loc[idx, 'height_change_m'] = h
    if d > 0.5:
        wp_alt.loc[idx, 'grade_pct'] = (h / d) * 100

# Cumulative distance
wp_alt['cum_dist_m'] = wp_alt['dist_m'].cumsum()

wp_alt[['wp', 'cum_dist_m', 'height_m', 'height_change_m', 'grade_pct']]

## Step 4: Visualize height profile

In [ ]:
from math import radians, sin, cos, sqrt, atan2  # already imported but just in case

# skip — already computed above

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Height relative to start
axes[0].plot(wp_alt['cum_dist_m'], wp_alt['height_m'], 'o-', label='Height from start')
axes[0].axhline(y=0, color='k', linewidth=0.5, linestyle='--')
axes[0].set_ylabel('Height (m)')
axes[0].set_title('Relative height along trail (zeroed to start)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grade between waypoints
axes[1].bar(wp_alt['cum_dist_m'], wp_alt['grade_pct'], width=wp_alt['dist_m'].mean() * 0.8, alpha=0.7, label='Grade %')
axes[1].axhline(y=0, color='k', linewidth=0.5, linestyle='--')
axes[1].set_ylabel('Grade (%)')
axes[1].set_title('Grade between waypoints')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[1].set_xlabel('Distance along trail (m)')
plt.tight_layout()
plt.show()

## Step 5: Export results

In [ ]:
output = wp_alt[['wp', 'lat', 'lng', 'cum_dist_m', 'alt', 'height_m',
                  'height_change_m', 'grade_pct', 'samples']].copy()

output.to_csv('height_profile.csv', index=False)
files.download('height_profile.csv')
print(f"Exported {len(output)} waypoints to height_profile.csv")